# 웨이퍼 결함 분류 포트폴리오 시각화 생성

LSWMD 데이터와 `modeling/artifacts`의 완료 실험 결과를 읽어 포트폴리오용 PNG를 생성한다. 모든 이미지는 650dpi로 저장한다. 기존 산출물 보호를 위해 같은 이름의 파일이 있으면 기본적으로 중단한다.

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from PIL import Image

OUTPUT_DPI = 650
ALLOW_REGENERATE = False
CLASS_ORDER = [
    'Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc',
    'Random', 'Scratch', 'Near-full', 'none',
]
CLASS_KO = {
    'Center': '중앙', 'Donut': '도넛', 'Edge-Loc': '가장자리 국소',
    'Edge-Ring': '가장자리 링', 'Loc': '국소', 'Random': '무작위',
    'Scratch': '스크래치', 'Near-full': '전면', 'none': '정상',
}
COLORS = {
    'navy': '#17324D', 'blue': '#2F6BFF', 'cyan': '#3CBCC3',
    'orange': '#F4A261', 'red': '#E76F51', 'green': '#2A9D8F',
    'gray': '#7A8793', 'light': '#EEF3F8', 'ink': '#17212B',
}

cwd = Path.cwd().resolve()
MODELING_ROOT = cwd if cwd.name == 'modeling' else cwd / 'modeling'
PROJECT_ROOT = MODELING_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'LSWMD.pkl'
ARTIFACT_ROOT = MODELING_ROOT / 'artifacts'
SUITE_DIR = ARTIFACT_ROOT / 'suites' / '4048895abf6e'
IMAGE_DIR = MODELING_ROOT / 'images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans'],
    'axes.unicode_minus': False,
    'axes.titleweight': 'bold',
    'axes.titlesize': 15,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

generated_paths: list[Path] = []


def save_figure(fig: plt.Figure, filename: str) -> Path:
    """그림을 고해상도 PNG로 저장하고 경로를 반환한다.

    Args:
        fig: 저장할 Matplotlib Figure.
        filename: `images` 폴더 아래의 PNG 파일명.

    Returns:
        저장한 이미지의 절대 경로.

    Raises:
        FileExistsError: 기존 파일 덮어쓰기가 허용되지 않은 경우.
    """
    path = IMAGE_DIR / filename
    if path.exists() and not ALLOW_REGENERATE:
        raise FileExistsError(
            f'기존 파일을 보호하기 위해 중단했습니다: {path}'
        )
    fig.savefig(
        path, dpi=OUTPUT_DPI, bbox_inches='tight',
        facecolor='white', metadata={'dpi': str(OUTPUT_DPI)},
    )
    plt.close(fig)
    generated_paths.append(path)
    return path


def add_box(
    ax: plt.Axes, xy: tuple[float, float], width: float, height: float,
    title: str, body: str, color: str,
) -> None:
    """좌표축에 설명 상자를 추가한다.

    Args:
        ax: 상자를 그릴 좌표축.
        xy: 상자 왼쪽 아래 좌표.
        width: 상자 너비.
        height: 상자 높이.
        title: 상자 제목.
        body: 상자 본문.
        color: 상자 테두리와 제목 색상.
    """
    box = FancyBboxPatch(
        xy, width, height, boxstyle='round,pad=0.018,rounding_size=0.025',
        linewidth=2, edgecolor=color, facecolor='white',
    )
    ax.add_patch(box)
    ax.text(
        xy[0] + width * 0.06, xy[1] + height * 0.80, title,
        fontsize=11.2, fontweight='bold', color=color, va='center',
    )
    ax.text(
        xy[0] + width * 0.06, xy[1] + height * 0.30, body,
        fontsize=8.8, color=COLORS['ink'], va='center', linespacing=1.4,
    )


## 1. 데이터와 실험 결과 로드

원본 811,457행 중 라벨이 존재하는 172,950행만 EDA에 사용한다. 실험 비교는 완료된 suite의 JSON/CSV 산출물만 사용한다.

In [2]:
def unwrap_label(value: object) -> object:
    """중첩 배열 라벨을 단일 값으로 변환한다.

    Args:
        value: 원본 failureType 값.

    Returns:
        중첩을 제거한 라벨 또는 결측값.
    """
    while isinstance(value, (list, tuple, np.ndarray)):
        if np.size(value) == 0:
            return None
        value = np.asarray(value, dtype=object).reshape(-1)[0]
    return value


raw = pd.read_pickle(DATA_PATH)
labels = raw['failureType'].map(unwrap_label)
labeled_mask = labels.notna() & labels.ne('')
frame = raw.loc[labeled_mask, ['waferMap', 'lotName']].copy()
frame['failureType'] = labels.loc[labeled_mask].astype(str)
frame['source_index'] = np.flatnonzero(labeled_mask.to_numpy())
frame.reset_index(drop=True, inplace=True)
raw_rows = len(raw)
del raw, labels, labeled_mask

shape_values = frame['waferMap'].map(lambda value: np.asarray(value).shape)
frame['height'] = shape_values.map(lambda value: value[0])
frame['width'] = shape_values.map(lambda value: value[1])
frame['aspect_ratio'] = frame['width'] / frame['height']
class_counts = frame['failureType'].value_counts().reindex(CLASS_ORDER)

suite_status = json.loads((SUITE_DIR / 'status.json').read_text('utf-8'))
final_summary = json.loads(
    (SUITE_DIR / 'final_summary.json').read_text('utf-8')
)
records = []
for experiment_id in suite_status['completed_ids']:
    experiment_dir = ARTIFACT_ROOT / experiment_id
    config = json.loads(
        (experiment_dir / 'config.json').read_text('utf-8')
    )['config']
    metrics = json.loads(
        (experiment_dir / 'metrics.json').read_text('utf-8')
    )
    records.append({
        'experiment_id': experiment_id, **config,
        'validation_macro_f1': metrics['validation']['macro_f1'],
        'validation_worst_f1': metrics['validation']['worst_class_f1'],
        'validation_accuracy': metrics['validation']['accuracy'],
        'parameters': metrics['parameters'],
        'best_epoch': metrics['best_epoch'],
        'epochs_run': metrics['epochs_run'],
        'elapsed_seconds': metrics['elapsed_seconds'],
    })
experiments = pd.DataFrame(records).set_index('experiment_id')

BASELINE_ID = 'small_cnn_17488d1a4881'
FINAL_ID = 'residual_cnn_483f70e6ae86'
baseline_f1 = experiments.loc[BASELINE_ID, 'validation_macro_f1']
final_validation_f1 = experiments.loc[FINAL_ID, 'validation_macro_f1']
test_macro_mean = final_summary['macro_f1_mean']
test_macro_std = final_summary['macro_f1_std']
test_accuracy_mean = np.mean([row['accuracy'] for row in final_summary['runs']])

print(f'원본/라벨 데이터: {raw_rows:,} / {len(frame):,}')
print(f'검증 Macro-F1: {baseline_f1:.4f} -> {final_validation_f1:.4f}')
print(f'테스트 Macro-F1: {test_macro_mean:.4f} ± {test_macro_std:.4f}')


원본/라벨 데이터: 811,457 / 172,950
검증 Macro-F1: 0.8582 -> 0.8851
테스트 Macro-F1: 0.8780 ± 0.0032


## 2. 프로젝트 요약과 이해관계자

표지용 KPI 카드, 이해관계자-페인포인트 연결도, 누수 방지형 실험 전략을 생성한다.

In [3]:
fig, ax = plt.subplots(figsize=(12, 5.2))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.text(
    0.04, 0.9, '웨이퍼 결함 분류: 정확도 착시를 넘어 소수 결함까지',
    fontsize=22, fontweight='bold', color=COLORS['navy'],
)
ax.text(
    0.04, 0.82,
    'Residual CNN과 종횡비 보존 전처리를 단계적으로 검증한 9클래스 분류 프로젝트',
    fontsize=12, color=COLORS['gray'],
)
cards = [
    ('검증 Macro-F1', f'{baseline_f1:.3f} → {final_validation_f1:.3f}',
     f'+{(final_validation_f1 - baseline_f1) * 100:.2f}%p', COLORS['blue']),
    ('테스트 Macro-F1', f'{test_macro_mean:.3f}',
     f'3 seed 표준편차 {test_macro_std:.3f}', COLORS['green']),
    ('테스트 정확도', f'{test_accuracy_mean * 100:.2f}%',
     '정상 85.24%의 영향 포함', COLORS['orange']),
    ('실험 통제', '11회 순차 실험',
     'test는 최종 설정 고정 후 1회', COLORS['red']),
]
for index, (title, value, note, color) in enumerate(cards):
    x = 0.04 + index * 0.24
    box = FancyBboxPatch(
        (x, 0.22), 0.21, 0.46,
        boxstyle='round,pad=0.015,rounding_size=0.025',
        facecolor='white', edgecolor=color, linewidth=2.2,
    )
    ax.add_patch(box)
    ax.text(x + 0.02, 0.59, title, fontsize=12, color=color, fontweight='bold')
    ax.text(x + 0.02, 0.43, value, fontsize=20, color=COLORS['ink'], fontweight='bold')
    ax.text(x + 0.02, 0.29, note, fontsize=9.5, color=COLORS['gray'])
ax.text(
    0.04, 0.08,
    '주의: 개선 폭은 동일 validation 비교, 최종 성능은 독립 test의 3-seed 평균이다.',
    fontsize=10, color=COLORS['red'],
)
save_figure(fig, '01_executive_summary.png')

fig, ax = plt.subplots(figsize=(13, 6.2))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('이해관계자의 페인포인트를 모델 KPI와 운영 의사결정으로 연결', pad=15)
stakeholders = [
    ('품질 엔지니어', '희귀 결함 누락\n육안 검사 편차', '클래스별 Recall\n혼동행렬', COLORS['red']),
    ('공정 엔지니어', '패턴별 원인 탐색 지연\n우선순위 불명확', '9개 패턴 분류\nLot 단위 집계', COLORS['orange']),
    ('MLOps 담당자', '후보 선택의 재현성 부족\n테스트 누수 위험', '고정 분할·seed\n실험 이력·승격 규칙', COLORS['blue']),
    ('관리자', '정확도 하나로는\n리스크 파악 곤란', 'Macro-F1\n비용 기반 KPI', COLORS['green']),
]
for index, (name, pain, metric, color) in enumerate(stakeholders):
    y = 0.72 - index * 0.21
    add_box(ax, (0.03, y), 0.24, 0.15, name, pain, color)
    add_box(ax, (0.38, y), 0.23, 0.15, '분석 기준', metric, color)
    add_box(
        ax, (0.73, y), 0.24, 0.15, '운영 의사결정',
        '자동 분류 / 검토 큐 /\n재학습·모니터링', color,
    )
    for start, end in [((0.28, y + 0.075), (0.37, y + 0.075)),
                       ((0.62, y + 0.075), (0.72, y + 0.075))]:
        ax.add_patch(FancyArrowPatch(
            start, end, arrowstyle='-|>', mutation_scale=14,
            color=COLORS['gray'], linewidth=1.5,
        ))
ax.text(0.15, 0.92, '누가 아픈가', fontsize=12, fontweight='bold', ha='center')
ax.text(0.495, 0.92, '무엇으로 측정하는가', fontsize=12, fontweight='bold', ha='center')
ax.text(0.85, 0.92, '어떤 행동으로 이어지는가', fontsize=12, fontweight='bold', ha='center')
save_figure(fig, '02_stakeholder_pain_point.png')

fig, ax = plt.subplots(figsize=(13, 5.8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('11회 순차 실험과 테스트 누수 방지 설계', pad=15)
stages = [
    ('1. 모델', '4개 후보', 'ResidualCNN', COLORS['blue']),
    ('2. 학습법', '4개 조합\n(1회 재사용)', 'CE · 증강 없음', COLORS['cyan']),
    ('3. 전처리', '3개 후보\n(1회 재사용)', '비율 보존 Pad', COLORS['orange']),
    ('4. Seed', '42 · 43 · 44\n(1회 재사용)', '검증으로 배포 seed 선택', COLORS['green']),
]
for index, (title, count, winner, color) in enumerate(stages):
    x = 0.03 + index * 0.235
    add_box(ax, (x, 0.43), 0.19, 0.33, title, f'{count}\n\n선택: {winner}', color)
    if index < len(stages) - 1:
        ax.add_patch(FancyArrowPatch(
            (x + 0.195, 0.595), (x + 0.23, 0.595),
            arrowstyle='-|>', mutation_scale=15, color=COLORS['gray'],
        ))
test_box = FancyBboxPatch(
    (0.17, 0.12), 0.66, 0.15,
    boxstyle='round,pad=0.02,rounding_size=0.025',
    facecolor=COLORS['light'], edgecolor=COLORS['navy'], linewidth=2,
)
ax.add_patch(test_box)
ax.text(
    0.5, 0.195,
    '최종 설정을 JSON으로 잠근 뒤에만 test 평가 → 3-seed 평균 ± 표준편차 보고',
    ha='center', va='center', fontsize=12, fontweight='bold',
    color=COLORS['navy'],
)
save_figure(fig, '07_experiment_strategy.png')


WindowsPath('C:/Users/ilove/OneDrive/07_SKALA/lecture/13_MLOps/MLOps_wafer/modeling/images/07_experiment_strategy.png')

## 3. EDA와 전처리

클래스 불균형, 대표 웨이퍼 맵, 원본 크기/종횡비, 리사이즈 방식 차이를 시각화한다.

In [4]:
fig, ax = plt.subplots(figsize=(11, 6))
colors = [COLORS['red']] * 8 + [COLORS['blue']]
bars = ax.bar(
    [CLASS_KO[name] for name in CLASS_ORDER], class_counts.values,
    color=colors, edgecolor='white',
)
ax.set_yscale('log')
ax.set_ylabel('샘플 수 (로그 축)')
ax.set_title('정상 클래스가 85.24%를 차지하는 극심한 불균형')
ax.grid(axis='y', alpha=0.25, which='both')
ax.spines[['top', 'right']].set_visible(False)
for bar, count in zip(bars, class_counts.values):
    share = count / len(frame) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2, count * 1.12,
        f'{count:,}\n({share:.2f}%)', ha='center', va='bottom', fontsize=8.5,
    )
ax.text(
    0.01, 0.03,
    f"정상 : Near-full = {class_counts['none'] / class_counts['Near-full']:.0f} : 1",
    transform=ax.transAxes, fontsize=10, color=COLORS['red'],
    bbox={'facecolor': 'white', 'edgecolor': COLORS['red'], 'boxstyle': 'round'},
)
fig.tight_layout()
save_figure(fig, '03_class_distribution.png')

representatives = {}
for class_name in CLASS_ORDER:
    subset = frame.loc[frame['failureType'].eq(class_name)].head(250)
    ratios = subset['waferMap'].map(
        lambda value: np.mean(np.asarray(value) == 2)
    )
    representatives[class_name] = subset.loc[
        (ratios - ratios.median()).abs().idxmin(), 'waferMap'
    ]
wafer_cmap = ListedColormap(['#17212B', '#F5F7FA', '#E63946'])
fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for ax, class_name in zip(axes.flat, CLASS_ORDER):
    wafer_map = np.asarray(representatives[class_name])
    ax.imshow(wafer_map, cmap=wafer_cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(
        f"{CLASS_KO[class_name]} ({class_name})\n{wafer_map.shape[0]}×{wafer_map.shape[1]}",
        fontsize=11,
    )
    ax.axis('off')
fig.suptitle('9개 클래스의 대표 웨이퍼 맵', fontsize=17, fontweight='bold')
fig.text(0.5, 0.02, '검정=영역 밖 · 흰색=정상 셀 · 빨강=불량 셀', ha='center')
fig.tight_layout(rect=(0, 0.04, 1, 0.96))
save_figure(fig, '04_wafer_examples.png')

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].hist(
    frame['aspect_ratio'], bins=45, color=COLORS['blue'], alpha=0.85,
)
axes[0].axvline(1, color=COLORS['red'], linestyle='--', label='정사각형')
axes[0].set_title('원본 종횡비 분포')
axes[0].set_xlabel('너비 / 높이')
axes[0].set_ylabel('샘플 수')
axes[0].legend()
axes[0].text(
    0.98, 0.92,
    f"비정사각형 {frame['width'].ne(frame['height']).mean() * 100:.1f}%",
    transform=axes[0].transAxes, ha='right', fontsize=11,
    bbox={'facecolor': 'white', 'edgecolor': COLORS['blue'], 'boxstyle': 'round'},
)
shape_counts = (
    frame.groupby(['height', 'width']).size().sort_values(ascending=False).head(15)
)
shape_labels = [f'{height}×{width}' for height, width in shape_counts.index]
axes[1].barh(shape_labels[::-1], shape_counts.values[::-1], color=COLORS['cyan'])
axes[1].set_title('빈도가 높은 원본 크기 Top 15')
axes[1].set_xlabel('샘플 수')
axes[1].grid(axis='x', alpha=0.2)
fig.suptitle(
    f'346개 원본 크기: 단순 정사각 리사이즈는 형상을 왜곡할 수 있다',
    fontsize=16, fontweight='bold',
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
save_figure(fig, '05_map_geometry.png')


def resize_fixed(wafer_map: np.ndarray, size: int = 64) -> np.ndarray:
    """종횡비를 무시하고 최근접 보간으로 리사이즈한다.

    Args:
        wafer_map: 원본 범주형 웨이퍼 맵.
        size: 출력 정사각형 한 변.

    Returns:
        리사이즈한 범주형 맵.
    """
    image = Image.fromarray(np.asarray(wafer_map, dtype=np.uint8))
    return np.asarray(image.resize((size, size), Image.Resampling.NEAREST))


def resize_pad(wafer_map: np.ndarray, size: int = 64) -> np.ndarray:
    """종횡비를 보존해 리사이즈하고 중앙 패딩한다.

    Args:
        wafer_map: 원본 범주형 웨이퍼 맵.
        size: 출력 정사각형 한 변.

    Returns:
        리사이즈와 중앙 패딩을 적용한 범주형 맵.
    """
    source = np.asarray(wafer_map, dtype=np.uint8)
    height, width = source.shape
    scale = size / max(height, width)
    new_height = max(1, round(height * scale))
    new_width = max(1, round(width * scale))
    image = Image.fromarray(source).resize(
        (new_width, new_height), Image.Resampling.NEAREST
    )
    padded = np.zeros((size, size), dtype=np.uint8)
    top = (size - new_height) // 2
    left = (size - new_width) // 2
    padded[top:top + new_height, left:left + new_width] = np.asarray(image)
    return padded


candidates = frame.loc[frame['failureType'].ne('none')].head(5000).copy()
distortion = np.abs(np.log(candidates['aspect_ratio']))
target_distortion = distortion.quantile(0.90)
sample_index = (distortion - target_distortion).abs().idxmin()
sample = np.asarray(candidates.loc[sample_index, 'waferMap'])
fixed = resize_fixed(sample)
padded = resize_pad(sample)
mask = (padded > 0).astype(np.uint8)
fig, axes = plt.subplots(1, 4, figsize=(13, 4))
panels = [
    (sample, f'원본\n{sample.shape[0]}×{sample.shape[1]}', wafer_cmap, 2),
    (fixed, 'Fixed resize\n64×64 · 형상 왜곡', wafer_cmap, 2),
    (padded, 'Resize + Pad\n64×64 · 비율 보존', wafer_cmap, 2),
    (mask, 'Pad + Mask의 mask 채널\n유효 영역=1', 'Greys', 1),
]
for ax, (image, title, cmap, vmax) in zip(axes, panels):
    ax.imshow(image, cmap=cmap, vmin=0, vmax=vmax, interpolation='nearest')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
fig.suptitle('세 전처리 후보가 웨이퍼 형상을 다루는 방식', fontsize=16, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.92))
save_figure(fig, '06_preprocessing_comparison.png')


WindowsPath('C:/Users/ilove/OneDrive/07_SKALA/lecture/13_MLOps/MLOps_wafer/modeling/images/06_preprocessing_comparison.png')

## 4. 모델링 선택 근거

모델, 손실/증강, 전처리를 동일 validation Macro-F1 기준으로 비교한다.

In [5]:
model_ids = [
    'small_cnn_17488d1a4881', 'spatial_cnn_eaa6f09b055c',
    'residual_cnn_8123daeff8cd', 'hybrid_cnn_3b52190d3bbe',
]
model_names = ['SmallCNN', 'SpatialCNN', 'ResidualCNN', 'HybridCNN']
model_rows = experiments.loc[model_ids]
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
bar_colors = [COLORS['gray'], COLORS['cyan'], COLORS['green'], COLORS['orange']]
bars = axes[0].bar(model_names, model_rows['validation_macro_f1'], color=bar_colors)
axes[0].set_ylim(0.84, 0.89)
axes[0].set_ylabel('Validation Macro-F1')
axes[0].set_title('모델 성능')
axes[0].grid(axis='y', alpha=0.2)
for bar, value in zip(bars, model_rows['validation_macro_f1']):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, value + 0.001, f'{value:.3f}',
        ha='center', fontsize=10,
    )
axes[1].scatter(
    model_rows['parameters'] / 1000, model_rows['validation_macro_f1'],
    s=model_rows['elapsed_seconds'] / 4, c=bar_colors, alpha=0.8,
    edgecolor='white', linewidth=1.5,
)
for name, (_, row) in zip(model_names, model_rows.iterrows()):
    axes[1].annotate(
        name, (row['parameters'] / 1000, row['validation_macro_f1']),
        xytext=(5, 6), textcoords='offset points', fontsize=9,
    )
axes[1].set_xlabel('파라미터 수 (천 개)')
axes[1].set_ylabel('Validation Macro-F1')
axes[1].set_title('성능-복잡도-학습시간 비교\n(원 크기=학습시간)')
axes[1].grid(alpha=0.2)
fig.suptitle('잔차 연결 모델이 1단계 우승', fontsize=16, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.92))
save_figure(fig, '08_model_comparison.png')

recipe_ids = [
    'residual_cnn_e19d7dd38d6c', 'residual_cnn_345f6ea69a42',
    'residual_cnn_8123daeff8cd', 'residual_cnn_ff67addb590f',
]
recipe_names = ['CE / 증강 없음', 'CE / 증강', 'Weighted CE / 증강 없음', 'Weighted CE / 증강']
recipe_rows = experiments.loc[recipe_ids]
fig, ax = plt.subplots(figsize=(10.5, 5.5))
recipe_colors = [COLORS['green'], COLORS['gray'], COLORS['gray'], COLORS['gray']]
bars = ax.barh(recipe_names[::-1], recipe_rows['validation_macro_f1'].values[::-1], color=recipe_colors[::-1])
ax.set_xlim(0.85, 0.89)
ax.set_xlabel('Validation Macro-F1')
ax.set_title('Weighted CE와 회전·반전 증강이 항상 이득은 아니었다')
ax.grid(axis='x', alpha=0.2)
for bar, value in zip(bars, recipe_rows['validation_macro_f1'].values[::-1]):
    ax.text(value + 0.0005, bar.get_y() + bar.get_height() / 2, f'{value:.3f}', va='center')
ax.text(
    0.01, 0.03, '선택: 기본 CE · 증강 없음', transform=ax.transAxes,
    color=COLORS['green'], fontweight='bold', fontsize=11,
)
fig.tight_layout()
save_figure(fig, '09_recipe_comparison.png')

preprocess_ids = [
    'residual_cnn_e19d7dd38d6c', 'residual_cnn_483f70e6ae86',
    'residual_cnn_4f3752a9111c',
]
preprocess_names = ['Fixed resize', 'Resize + Pad', 'Resize + Pad + Mask']
preprocess_rows = experiments.loc[preprocess_ids]
fig, ax = plt.subplots(figsize=(9.5, 5.5))
preprocess_colors = [COLORS['gray'], COLORS['green'], COLORS['orange']]
bars = ax.bar(preprocess_names, preprocess_rows['validation_macro_f1'], color=preprocess_colors)
ax.set_ylim(0.84, 0.895)
ax.set_ylabel('Validation Macro-F1')
ax.set_title('종횡비 보존 Pad는 개선, 추가 Mask 채널은 악화')
ax.grid(axis='y', alpha=0.2)
fixed_value = preprocess_rows.iloc[0]['validation_macro_f1']
for bar, value in zip(bars, preprocess_rows['validation_macro_f1']):
    delta = (value - fixed_value) * 100
    label = f'{value:.3f}' if abs(delta) < 1e-9 else f'{value:.3f}\n({delta:+.2f}%p)'
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.001, label, ha='center', fontsize=10)
fig.tight_layout()
save_figure(fig, '10_preprocessing_performance.png')


WindowsPath('C:/Users/ilove/OneDrive/07_SKALA/lecture/13_MLOps/MLOps_wafer/modeling/images/10_preprocessing_performance.png')

## 5. 평가와 비즈니스 해석

학습 곡선, seed 안정성, 클래스별 성능, 혼동행렬, 운영 액션 매트릭스를 생성한다.

In [6]:
history = pd.read_csv(ARTIFACT_ROOT / FINAL_ID / 'history.csv')
best_epoch = int(experiments.loc[FINAL_ID, 'best_epoch'])
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
axes[0].plot(history['epoch'], history['validation_macro_f1'], color=COLORS['blue'], linewidth=2, label='Validation Macro-F1')
axes[0].plot(history['epoch'], history['best_validation_macro_f1'], color=COLORS['green'], linestyle='--', label='Best so far')
axes[0].axvline(best_epoch, color=COLORS['red'], linestyle=':', label=f'최고 epoch {best_epoch}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Macro-F1')
axes[0].set_title('검증 Macro-F1')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.2)
axes[1].plot(history['epoch'], history['train_loss'], color=COLORS['gray'], label='Train loss')
axes[1].plot(history['epoch'], history['validation_loss'], color=COLORS['orange'], label='Validation loss')
axes[1].axvline(best_epoch, color=COLORS['red'], linestyle=':')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('학습·검증 Loss')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.2)
fig.suptitle('배포 후보(seed 42)의 학습 이력', fontsize=16, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.93))
save_figure(fig, '11_learning_curve.png')

final_ids = [row['experiment_id'] for row in final_summary['runs']]
seeds = [row['seed'] for row in final_summary['runs']]
validation_scores = experiments.loc[final_ids, 'validation_macro_f1'].to_numpy()
test_scores = np.array([row['macro_f1'] for row in final_summary['runs']])
x = np.arange(len(seeds))
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.plot(x, validation_scores, 'o-', color=COLORS['blue'], linewidth=2, markersize=8, label='Validation')
ax.plot(x, test_scores, 'o-', color=COLORS['green'], linewidth=2, markersize=8, label='Test')
ax.axhline(test_scores.mean(), color=COLORS['green'], linestyle='--', alpha=0.7, label=f'Test 평균 {test_scores.mean():.3f}')
ax.fill_between(x, test_scores.mean() - test_scores.std(ddof=1), test_scores.mean() + test_scores.std(ddof=1), color=COLORS['green'], alpha=0.12)
ax.set_xticks(x, [str(seed) for seed in seeds])
ax.set_xlabel('Random seed')
ax.set_ylabel('Macro-F1')
ax.set_ylim(0.85, 0.90)
ax.set_title(f'최종 성능 안정성: Test {test_scores.mean():.3f} ± {test_scores.std(ddof=1):.3f}')
ax.legend()
ax.grid(alpha=0.2)
for index, value in enumerate(test_scores):
    ax.text(index, value - 0.004, f'{value:.3f}', ha='center', color=COLORS['green'])
fig.tight_layout()
save_figure(fig, '12_seed_stability.png')

class_f1 = pd.DataFrame({
    row['seed']: {name: values['f1'] for name, values in row['per_class'].items()}
    for row in final_summary['runs']
}).reindex(CLASS_ORDER)
class_recall = pd.DataFrame({
    row['seed']: {name: values['recall'] for name, values in row['per_class'].items()}
    for row in final_summary['runs']
}).reindex(CLASS_ORDER)
support = pd.Series(
    {name: final_summary['runs'][0]['per_class'][name]['support'] for name in CLASS_ORDER}
)
means = class_f1.mean(axis=1)
stds = class_f1.std(axis=1, ddof=1)
order = means.sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(13, 6.2), gridspec_kw={'width_ratios': [1.45, 1]})
performance_colors = [COLORS['red'] if means[name] < 0.8 else COLORS['green'] for name in order]
axes[0].barh(
    [CLASS_KO[name] for name in order], means.loc[order],
    xerr=stds.loc[order], color=performance_colors, alpha=0.9, capsize=3,
)
axes[0].set_xlim(0.65, 1.01)
axes[0].set_xlabel('Test F1 (3-seed 평균 ± 표준편차)')
axes[0].set_title('클래스별 F1')
axes[0].grid(axis='x', alpha=0.2)
for index, name in enumerate(order):
    axes[0].text(means[name] + 0.008, index, f'{means[name]:.3f}', va='center', fontsize=9)
axes[1].barh(
    [CLASS_KO[name] for name in order], support.loc[order], color=COLORS['blue'], alpha=0.85,
)
axes[1].set_xscale('log')
axes[1].set_xlabel('클래스별 test support (로그 축)')
axes[1].set_title('해석의 근거가 되는 표본 수')
axes[1].grid(axis='x', alpha=0.2, which='both')
for index, name in enumerate(order):
    axes[1].text(support[name] * 1.08, index, f'{support[name]:,}', va='center', fontsize=9)
fig.suptitle('평균 성능과 표본 크기를 함께 봐야 한다', fontsize=16, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.93))
save_figure(fig, '13_class_performance.png')

confusion = pd.DataFrame(0, index=CLASS_ORDER, columns=CLASS_ORDER, dtype=int)
for experiment_id in final_ids:
    predictions = pd.read_csv(
        ARTIFACT_ROOT / experiment_id / 'test_predictions.csv',
        usecols=['true_label', 'predicted_label'],
    )
    confusion += pd.crosstab(
        predictions['true_label'], predictions['predicted_label']
    ).reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0)
normalized = confusion.div(confusion.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(normalized, cmap='Blues', vmin=0, vmax=100)
ax.set_xticks(range(9), [CLASS_KO[name] for name in CLASS_ORDER], rotation=35, ha='right')
ax.set_yticks(range(9), [CLASS_KO[name] for name in CLASS_ORDER])
ax.set_xlabel('예측 클래스')
ax.set_ylabel('실제 클래스')
ax.set_title('3-seed 합산 Test 혼동행렬 (행 기준 정규화)')
for row in range(9):
    for column in range(9):
        value = normalized.iloc[row, column]
        if value >= 0.5 or row == column:
            ax.text(
                column, row, f'{value:.1f}', ha='center', va='center',
                fontsize=8, color='white' if value > 55 else COLORS['ink'],
            )
colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
colorbar.set_label('비율 (%)')
fig.tight_layout()
save_figure(fig, '14_confusion_matrix.png')

mean_recall = class_recall.mean(axis=1)
actions = []
for class_name in CLASS_ORDER:
    if class_name == 'none' or mean_recall[class_name] >= 0.95:
        action = '자동 분류 + 표본 감사'
    elif mean_recall[class_name] >= 0.85:
        action = '자동 분류 + 추세 모니터링'
    else:
        action = '저신뢰 예측 검토 큐 우선'
    if support[class_name] < 100:
        action = '표본 확충 전 보수적 검토'
    actions.append([
        f'{CLASS_KO[class_name]} ({class_name})',
        f'{mean_recall[class_name]:.3f}',
        f'{(1 - mean_recall[class_name]) * 100:.1f}',
        f'{support[class_name]:,}',
        action,
    ])
fig, ax = plt.subplots(figsize=(13, 6.3))
ax.axis('off')
ax.set_title('클래스별 운영 액션 매트릭스', pad=18)
table = ax.table(
    cellText=actions,
    colLabels=['클래스', '평균 Recall', '100건당 예상 누락', 'Test support', '권고 운영'],
    colWidths=[0.22, 0.12, 0.16, 0.13, 0.31],
    cellLoc='center', loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1, 1.65)
for (row, column), cell in table.get_celld().items():
    cell.set_edgecolor('white')
    if row == 0:
        cell.set_facecolor(COLORS['navy'])
        cell.get_text().set_color('white')
        cell.get_text().set_weight('bold')
    else:
        class_name = CLASS_ORDER[row - 1]
        cell.set_facecolor('#FDECEA' if mean_recall[class_name] < 0.8 else COLORS['light'])
ax.text(
    0.5, 0.06,
    '※ 100건당 예상 누락은 (1 − Recall)×100의 모델 지표 환산값이며 실제 비용·발생률은 별도 입력이 필요하다.',
    transform=ax.transAxes, ha='center', fontsize=9, color=COLORS['gray'],
)
fig.tight_layout()
save_figure(fig, '15_business_action_matrix.png')


WindowsPath('C:/Users/ilove/OneDrive/07_SKALA/lecture/13_MLOps/MLOps_wafer/modeling/images/15_business_action_matrix.png')

## 6. 산출물 검증

생성 파일 수, 픽셀 크기, DPI 메타데이터를 확인한다. PNG의 물리 해상도 환산 오차를 고려해 650dpi로 저장했으므로 600dpi 이상이어야 한다.

In [7]:
checks = []
for path in generated_paths:
    with Image.open(path) as image:
        dpi = image.info.get('dpi', (0, 0))
        checks.append({
            '파일': path.name, '너비(px)': image.width, '높이(px)': image.height,
            'DPI-X': round(dpi[0], 2), 'DPI-Y': round(dpi[1], 2),
            '600dpi 이상': min(dpi) >= 600,
        })
checks_frame = pd.DataFrame(checks)
if len(checks_frame) != 15 or not checks_frame['600dpi 이상'].all():
    raise RuntimeError('이미지 수 또는 DPI 검증에 실패했습니다.')
checks_frame


,파일,너비(px),높이(px),DPI-X,DPI-Y,600dpi 이상
0,01_executive_summary.png,6175,2732,650.01,650.01,True
1,02_stakeholder_pain_point.png,6678,3470,650.01,650.01,True
2,07_experiment_strategy.png,6678,3270,650.01,650.01,True
3,03_class_distribution.png,7071,3836,650.01,650.01,True
4,04_wafer_examples.png,6042,5767,650.01,650.01,True
5,05_map_geometry.png,7728,3534,650.01,650.01,True
6,06_preprocessing_comparison.png,8018,2580,650.01,650.01,True
7,08_model_comparison.png,7712,3345,650.01,650.01,True
8,09_recipe_comparison.png,6764,3502,650.01,650.01,True
9,10_preprocessing_performance.png,6104,3507,650.01,650.01,True
